# Overview

This notebook is to explore how to add dependencies between issues using GitHub REST API.

Once I figure how to do it, I will add dependencies to issues that are already created in the repo.


# Setup

In [38]:
import requests
import os
import time

In [39]:
import data360_ops as dops

In [40]:
def load_token(file_path, key="WB_DECIS_TOKEN"):
    try:
        with open(file_path, "r") as f:
            for line in f:
                if "=" in line:
                    temp_key, value = line.strip().split("=", 1)
                    if temp_key.strip() == key:
                        return value.strip()
        raise ValueError("The token key you provided was not found")
    except FileNotFoundError as e:
        raise FileNotFoundError(f"File not found... {e}")

In [41]:
token = load_token("../../github.token")
owner = "WB-DECIS"
repo = "testing_issues"

# Extract list of datatasets ids

In [42]:
existing_dataset_ids = dops.github.extract_list_dataset_ids(
		token=token,
		owner=owner,
		repo=repo,
		label="Dataset"
	)

Retrieving issues from GitHub...
Extracting dataset IDs from issues...


In [43]:
existing_dataset_ids

{'TESTEST',
 'Test2',
 'Test3',
 'Testing source pipeline',
 'Testing subissues',
 'WB_ASDF',
 'WB_NEW',
 'WB_TEST1 - WB TEST 1 using API',
 'WB_TEST_111',
 'WB_TEST_2',
 'WB_TEST_3',
 'WB_TEST_4',
 'WB_TEST_5',
 'WB_TEST_6',
 'WB_TEST_7',
 'WB_TEST_8',
 'database_id'}

## Create issues

In [44]:
dataset_id = 'WB_ABC_1'
dataset_name = 'Test Dataset ABC 1'

In [45]:
issues = dops.github.create_issues(
			dataset_id=dataset_id,
			dataset_name=dataset_name,
			token=token,
			owner=owner,
			repo=repo
		)

[WB_ABC_1] - Test Dataset ABC 1 - ['Dataset']
Issue created successfully!
[WB_ABC_1] - Collection module - Developer - ['Collection', 'Task']
Issue created successfully!
[WB_ABC_1] - Data modeling - Curator - ['Modeling', 'Task']
Issue created successfully!
[WB_ABC_1] - Metadata elements creation - Curator - ['Metadata elements', 'Task']
Issue created successfully!
[WB_ABC_1] - Processing module - Developer - ['Processing', 'Task']
Issue created successfully!
[WB_ABC_1] - Pipeline to prod - Lead/Operations - ['Pip. to prod', 'Task']
Issue created successfully!
[WB_ABC_1] - Referential Metadata review - Metadata manager - ['Metadata review', 'Task']
Issue created successfully!
[WB_ABC_1] - Data and Metadata Approval - Curator - ['Meta-data approval', 'Task']
Issue created successfully!
[WB_ABC_1] - Schedule pipeline - Lead/Operations - ['Ops', 'Task']
Issue created successfully!
[WB_ABC_1] - Maintenance - Lead/Operations - ['Maintenance', 'Task']
Issue created successfully!


## Create subissues

In [46]:
# Add subissues
added = dops.github.add_subissues(
	created_issues=issues,
	token=token,
	owner=owner,
	repo=repo
)

Adding task [WB_ABC_1] - Collection module - Developer to dataset issue #335
Subissue added successfully!
Adding task [WB_ABC_1] - Data modeling - Curator to dataset issue #335
Subissue added successfully!
Adding task [WB_ABC_1] - Metadata elements creation - Curator to dataset issue #335
Subissue added successfully!
Adding task [WB_ABC_1] - Processing module - Developer to dataset issue #335
Subissue added successfully!
Adding task [WB_ABC_1] - Pipeline to prod - Lead/Operations to dataset issue #335
Subissue added successfully!
Adding task [WB_ABC_1] - Referential Metadata review - Metadata manager to dataset issue #335
Subissue added successfully!
Adding task [WB_ABC_1] - Data and Metadata Approval - Curator to dataset issue #335
Subissue added successfully!
Adding task [WB_ABC_1] - Schedule pipeline - Lead/Operations to dataset issue #335
Subissue added successfully!
Adding task [WB_ABC_1] - Maintenance - Lead/Operations to dataset issue #335
Subissue added successfully!


## Search issues by string pattern

In [13]:
# Set up headers with the token
headers = {
    "Authorization": f"Bearer {token}",
    "Accept": "application/vnd.github+json",
}

In [14]:
search_term = "WB_TEST_111"  # The pattern you're searching for

In [15]:
# GitHub Search API endpoint
url = "https://api.github.com/search/issues"

In [16]:
# Query parameters
params = {
    "q": f"{search_term} in:title repo:{owner}/{repo} is:issue",
}


In [17]:
# Make the request
response = requests.get(url, headers=headers, params=params)

# Check response
if response.status_code == 200:
    issues = response.json()["items"]
    for issue in issues:
        print(f"- #{issue['number']}: {issue['title']} ({issue['html_url']})")
else:
    print(f"Error: {response.status_code}")
    print(response.text)


- #334: [WB_TEST_111] - Maintenance - Lead/Operations (https://github.com/WB-DECIS/testing_issues/issues/334)
- #329: [WB_TEST_111] - Processing module - Developer (https://github.com/WB-DECIS/testing_issues/issues/329)
- #327: [WB_TEST_111] - Data modeling - Curator (https://github.com/WB-DECIS/testing_issues/issues/327)
- #326: [WB_TEST_111] - Collection module - Developer (https://github.com/WB-DECIS/testing_issues/issues/326)
- #333: [WB_TEST_111] - Schedule pipeline - Lead/Operations (https://github.com/WB-DECIS/testing_issues/issues/333)
- #328: [WB_TEST_111] - Metadata elements creation - Curator (https://github.com/WB-DECIS/testing_issues/issues/328)
- #331: [WB_TEST_111] - Referencial Metadata review - Metadata manager (https://github.com/WB-DECIS/testing_issues/issues/331)
- #332: [WB_TEST_111] - Data and Metadata Approval - Curator (https://github.com/WB-DECIS/testing_issues/issues/332)
- #330: [WB_TEST_111] - Pipeline to prod - Lead/Operations (https://github.com/WB-DECIS/t

In [20]:
issues_found = []
for issue in response.json().get("items", []):
    issues_found.append({
		"id": issue.get("id"),
		"node_id": issue.get("node_id"),
		"number": issue.get("number"),
		"title": issue.get("title"),
        "labels": [label.get("name") for label in issue.get("labels", [])],
		"url": issue.get("html_url"),
		"state": issue.get("state"),
	})
issues_found

[{'id': 3648994355,
  'node_id': 'I_kwDOMdlsPs7Zfzwz',
  'number': 334,
  'title': '[WB_TEST_111] - Maintenance - Lead/Operations',
  'labels': ['Task', 'Maintenance'],
  'url': 'https://github.com/WB-DECIS/testing_issues/issues/334',
  'state': 'open'},
 {'id': 3648994074,
  'node_id': 'I_kwDOMdlsPs7Zfzsa',
  'number': 329,
  'title': '[WB_TEST_111] - Processing module - Developer',
  'labels': ['Task', 'Processing'],
  'url': 'https://github.com/WB-DECIS/testing_issues/issues/329',
  'state': 'open'},
 {'id': 3648993960,
  'node_id': 'I_kwDOMdlsPs7Zfzqo',
  'number': 327,
  'title': '[WB_TEST_111] - Data modeling - Curator',
  'labels': ['Task', 'Modeling'],
  'url': 'https://github.com/WB-DECIS/testing_issues/issues/327',
  'state': 'open'},
 {'id': 3648993906,
  'node_id': 'I_kwDOMdlsPs7Zfzpy',
  'number': 326,
  'title': '[WB_TEST_111] - Collection module - Developer',
  'labels': ['Task', 'Collection'],
  'url': 'https://github.com/WB-DECIS/testing_issues/issues/326',
  'state': 

In [21]:
# Create subsets by tasks, epics and datasets labels
datasets = [issue for issue in issues_found if "Dataset" in issue["labels"]]
# epics = [issue for issue in created_issues if "Epic" in issue["labels"]]
tasks = [issue for issue in issues_found if "Task" in issue["labels"]]

In [22]:
datasets

[{'id': 3648993856,
  'node_id': 'I_kwDOMdlsPs7ZfzpA',
  'number': 325,
  'title': '[WB_TEST_111] - WB TEST 111 simplified version',
  'labels': ['Dataset'],
  'url': 'https://github.com/WB-DECIS/testing_issues/issues/325',
  'state': 'open'}]

In [23]:
tasks

[{'id': 3648994355,
  'node_id': 'I_kwDOMdlsPs7Zfzwz',
  'number': 334,
  'title': '[WB_TEST_111] - Maintenance - Lead/Operations',
  'labels': ['Task', 'Maintenance'],
  'url': 'https://github.com/WB-DECIS/testing_issues/issues/334',
  'state': 'open'},
 {'id': 3648994074,
  'node_id': 'I_kwDOMdlsPs7Zfzsa',
  'number': 329,
  'title': '[WB_TEST_111] - Processing module - Developer',
  'labels': ['Task', 'Processing'],
  'url': 'https://github.com/WB-DECIS/testing_issues/issues/329',
  'state': 'open'},
 {'id': 3648993960,
  'node_id': 'I_kwDOMdlsPs7Zfzqo',
  'number': 327,
  'title': '[WB_TEST_111] - Data modeling - Curator',
  'labels': ['Task', 'Modeling'],
  'url': 'https://github.com/WB-DECIS/testing_issues/issues/327',
  'state': 'open'},
 {'id': 3648993906,
  'node_id': 'I_kwDOMdlsPs7Zfzpy',
  'number': 326,
  'title': '[WB_TEST_111] - Collection module - Developer',
  'labels': ['Task', 'Collection'],
  'url': 'https://github.com/WB-DECIS/testing_issues/issues/326',
  'state': 

In [26]:
tags = ['Modeling', 'Metadata elements', 'Processing', 'Pip. to prod', 'Metadata review', 'Meta-data approval', 'Ops', 'Maintenance']

In [28]:
len(tags)

8

In [ ]:
for i in range(len(tags)):
	tag = tags[i]
	if tag == 'Modeling':
		continue
	else:
		previous_tag = tags[i - 1]
		# Extract issue number based on tag and then the number
		issue_number = [i for i in tasks if tag in i['labels']][0]['number']
		# Extract previous issue id based on tag
		previous_issue_id = [i for i in tasks if previous_tag in i['labels']][0]['id']
		dependency_url = f"https://api.github.com/repos/{owner}/{repo}/issues/{issue_number}/dependencies/blocked_by"
		params = {
			"issue_id": previous_issue_id
		}
		# Make the request
		response = requests.post(dependency_url, headers=headers, json=params)

		# Check the response
		if response.status_code in [200, 201]:
			print(f"{tag}: 'Blocked by' dependency added successfully!")
			# print(response.json())
		else:
			print(f"Could not add dependency: {response.status_code}")
			print(response.text)
		time.sleep(0.1)

Metadata elements: Blocked by dependency added successfully!
Processing: Blocked by dependency added successfully!
Pip. to prod: Blocked by dependency added successfully!
Metadata review: Blocked by dependency added successfully!
Meta-data approval: Blocked by dependency added successfully!
Ops: Blocked by dependency added successfully!
Maintenance: Blocked by dependency added successfully!


In [36]:
tag = "Metadata elements"
issue = [i for i in tasks if tag in i['labels']][0]['number']
issue

328

In [ ]:
for tag in tags:
	if tag == 'Modeling':
		continue
	else:
		

# Functions for production

In [48]:
def add_dependencies(
		created_issues: list,
		token: str, 
		owner: str, 
		repo: str
	):
	"""Function to add dependencies to created issues in a GitHub repo.

	This function add dependencies between issues.
	
	Args:
		created_issues (list): List of created issues with their details.
		token (str): Personal access token for GitHub API authentication.
		owner (str): GitHub repository owner.
		repo (str): GitHub repository name.
	Returns:
		Bool: True if dependencies were added successfully, False otherwise.
	"""
	# Define order for dependencies
	# First element of the list won't have any dependencies
	tags = ['Modeling', 'Metadata elements', 'Processing', 'Pip. to prod', 
		 'Metadata review', 'Meta-data approval', 'Ops', 'Maintenance']

	# Set up headers with the token
	headers = {
		"Authorization": f"Bearer {token}",
		"Accept": "application/vnd.github+json",
	}

	# Create subsets by tasks, epics and datasets labels
	datasets = [issue for issue in created_issues if "Dataset" in issue["labels"]]
	# epics = [issue for issue in created_issues if "Epic" in issue["labels"]]
	tasks = [issue for issue in created_issues if "Task" in issue["labels"]]

	for i in range(len(tags)):
		tag = tags[i]
		if tag == 'Modeling':
			continue
		else:
			previous_tag = tags[i - 1]
			# Extract issue number based on tag and then the number
			issue_number = [i for i in tasks if tag in i['labels']][0]['number']
			# Extract previous issue id based on tag
			previous_issue_id = [i for i in tasks if previous_tag in i['labels']][0]['id']
			dependency_url = f"https://api.github.com/repos/{owner}/{repo}/issues/{issue_number}/dependencies/blocked_by"
			params = {
				"issue_id": previous_issue_id
			}
			# Make the request
			response = requests.post(dependency_url, headers=headers, json=params)

			# Check the response
			if response.status_code in [200, 201]:
				print(f"{tag}: 'Blocked by' dependency added successfully!")
				# print(response.json())
			else:
				print(f"Could not add dependency: {response.status_code}")
				print(response.text)
			time.sleep(0.1)
		# End if
	# End for loop
	return True

In [49]:
dependencies = add_dependencies(
		created_issues=issues,
		token=token,
		owner=owner,
		repo=repo
	)

Metadata elements: 'Blocked by' dependency added successfully!
Processing: 'Blocked by' dependency added successfully!
Pip. to prod: 'Blocked by' dependency added successfully!
Metadata review: 'Blocked by' dependency added successfully!
Meta-data approval: 'Blocked by' dependency added successfully!
Ops: 'Blocked by' dependency added successfully!
Maintenance: 'Blocked by' dependency added successfully!
